# Python Seminar No.7: Applicability Domain

`boston.csv` の `MEDV` を目的変数として回帰モデルを作成し、Hold-Out検証で得られたテストデータが適用ドメイン (AD) 内か外かを判定します。

標準化した説明変数空間における **kNN平均距離AD** を基本にしつつ、独自性として **Mahalanobis距離AD** も追加します。2種類のAD判定を組み合わせ、予測値の信頼度を `high` / `medium` / `low` の3段階で可視化します。


## 指定スライドとの対応

- 使用スライド: `適用ドメイン.pptx`
- スライド1: ADの定義、kNN平均距離、95パーセンタイルしきい値、AD内外判定、課題内容。
- スライド2: 可視化例。学習データ間の平均距離分布としきい値、テストデータのk近傍距離と予測誤差。
- スライド3: 実データでAD内外を色分けして予測信頼性を考える例。

## 1. ライブラリの読み込み

In [ ]:
# OS操作やファイルパスを扱うためのライブラリを読み込みます。
import os
from pathlib import Path

# 現在の作業フォルダをプロジェクトフォルダとして保存します。
PROJECT_DIR = Path.cwd()

# matplotlibのキャッシュ保存先をノートブック内のフォルダに作ります。
(PROJECT_DIR / ".matplotlib-cache").mkdir(exist_ok=True)

# matplotlibがキャッシュを書き込めるように環境変数を設定します。
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_DIR / ".matplotlib-cache"))

# 並列計算時のCPU数に関する警告を避けるため、使用CPU数を1に固定します。
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

# グラフ描画、数値計算、表データ処理に使う基本ライブラリを読み込みます。
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Mahalanobis距離で使う共分散推定モデルを読み込みます。
from sklearn.covariance import LedoitWolf

# 目的変数を変換してから回帰するための道具を読み込みます。
from sklearn.compose import TransformedTargetRegressor

# 比較用のベースラインモデルを読み込みます。
from sklearn.dummy import DummyRegressor

# 複数のアンサンブル回帰モデルを読み込みます。
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
    StackingRegressor,
    VotingRegressor,
)

# 線形回帰と正則化つき線形回帰モデルを読み込みます。
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge

# 予測性能を評価する指標を読み込みます。
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# データを学習用とテスト用に分ける関数を読み込みます。
from sklearn.model_selection import train_test_split

# k近傍回帰と、AD計算に使う近傍探索を読み込みます。
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors

# 標準化やロバストスケーリングに使う前処理を読み込みます。
from sklearn.preprocessing import PowerTransformer, RobustScaler, StandardScaler

# サポートベクター回帰と決定木回帰を読み込みます。
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# seabornとmatplotlibの見た目を設定します。
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# 乱数を使う処理で同じ結果を再現できるように、乱数シードを固定します。
RANDOM_STATE = 42


## スライド対応: スライド1

- ADとは、学習データに近い特徴量空間での予測を信頼し、離れた外挿領域の予測信頼性を低く見るための考え方。
- このノートブックでは、スライド1の課題どおり `boston.csv` の `MEDV` を目的変数にしてHold-Out検証を行う。

## 2. データの読み込み

`MEDV` を目的変数、それ以外の列を説明変数として扱います。

In [ ]:
# 読み込むCSVファイルのパスを指定します。
data_path = Path("boston.csv")

# 先頭列をインデックスとしてCSVファイルを読み込みます。
df = pd.read_csv(data_path, index_col=0)

# データの先頭5行を表示して、列名や値の形式を確認します。
display(df.head())

# データ数、列数、欠損値数を確認します。
print(f"データ数: {df.shape[0]}")
print(f"列数: {df.shape[1]}")
print(f"欠損値数: {int(df.isna().sum().sum())}")


In [ ]:
# 目的変数として使う列名を指定します。
target_col = "MEDV"

# MEDV以外の列を説明変数Xとして取り出します。
X = df.drop(columns=target_col)

# MEDV列を目的変数yとして取り出します。
y = df[target_col]

# Hold-Out検証のため、データを学習用75%とテスト用25%に分割します。
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
)

# 分割後の学習データ数とテストデータ数を確認します。
print(f"学習データ数: {len(X_train)}")
print(f"テストデータ数: {len(X_test)}")


## スライド対応: スライド1

- `MEDV` を目的変数、残りの13列を説明変数として使う。
- Hold-Out検証として、学習データ75%、テストデータ25%に分割する。

## 3. 前処理と独自モデル比較

距離計算にスケールの影響が出るため、AD判定用には標準化した説明変数を使います。一方、回帰モデル側では標準化データとRobustScalerデータの両方を用意し、単体モデルだけでなくVoting/Stackingも試します。

このノートブックでは、例ファイルと違い、最終的に **Stacking回帰** も候補に入れて予測値を選びます。

In [ ]:
# StandardScalerで、各説明変数の平均を0、標準偏差を1にそろえます。
standard_scaler = StandardScaler()
X_train_scaled = standard_scaler.fit_transform(X_train)
X_test_scaled = standard_scaler.transform(X_test)

# RobustScalerで、外れ値の影響を受けにくいスケーリングも用意します。
robust_scaler = RobustScaler()
X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

# 木構造ベースのモデルをまとめて定義します。
# 木系モデルは特徴量スケールの影響を受けにくいため、元の説明変数を使います。
tree_models = {
    "Random Forest tuned": RandomForestRegressor(
        n_estimators=700,
        max_features="sqrt",
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Extra Trees tuned": ExtraTreesRegressor(
        n_estimators=700,
        max_features=0.75,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "Gradient Boosting tuned": GradientBoostingRegressor(
        n_estimators=350,
        learning_rate=0.035,
        max_depth=3,
        subsample=0.85,
        random_state=RANDOM_STATE,
    ),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        learning_rate=0.055,
        max_iter=400,
        l2_regularization=0.03,
        random_state=RANDOM_STATE,
    ),
}

# スケーリングしたデータで使うモデルをまとめて定義します。
# 線形モデル、SVR、kNNは特徴量のスケールの影響を受けやすいです。
scaled_models = {
    "Dummy (mean)": DummyRegressor(strategy="mean"),
    "Huber robust linear": HuberRegressor(epsilon=1.5, alpha=0.0005, max_iter=1000),
    "Elastic Net robust scale": ElasticNet(alpha=0.005, l1_ratio=0.25, max_iter=20000, random_state=RANDOM_STATE),
    "SVR + log target": TransformedTargetRegressor(
        regressor=SVR(kernel="rbf", C=20, gamma="scale", epsilon=0.05),
        func=np.log1p,
        inverse_func=np.expm1,
    ),
    "kNN distance weighted": KNeighborsRegressor(n_neighbors=7, weights="distance"),
}

# 複数モデルの平均的な予測を使うVoting回帰モデルを定義します。
voting_model = VotingRegressor(
    estimators=[
        ("et", ExtraTreesRegressor(n_estimators=400, max_features=0.75, random_state=RANDOM_STATE, n_jobs=-1)),
        ("gbr", GradientBoostingRegressor(n_estimators=300, learning_rate=0.04, subsample=0.85, random_state=RANDOM_STATE)),
        ("svr", TransformedTargetRegressor(regressor=SVR(C=20, epsilon=0.05), func=np.log1p, inverse_func=np.expm1)),
    ]
)

# 複数モデルの予測をさらにRidge回帰でまとめるStacking回帰モデルを定義します。
stacking_model = StackingRegressor(
    estimators=[
        ("et", ExtraTreesRegressor(n_estimators=500, max_features=0.75, random_state=RANDOM_STATE, n_jobs=-1)),
        ("gbr", GradientBoostingRegressor(n_estimators=350, learning_rate=0.035, subsample=0.85, random_state=RANDOM_STATE)),
        ("huber", HuberRegressor(epsilon=1.5, alpha=0.0005, max_iter=1000)),
    ],
    final_estimator=Ridge(alpha=1.0),
    passthrough=True,
    cv=5,
    n_jobs=-1,
)

# モデル名、モデル本体、学習データ、テストデータを1つのリストにまとめます。
model_specs = []
for name, candidate_model in tree_models.items():
    model_specs.append((name, candidate_model, X_train, X_test))
for name, candidate_model in scaled_models.items():
    data_train = X_train_robust if "robust" in name.lower() else X_train_scaled
    data_test = X_test_robust if "robust" in name.lower() else X_test_scaled
    model_specs.append((name, candidate_model, data_train, data_test))
model_specs.extend(
    [
        ("Voting ensemble", voting_model, X_train_scaled, X_test_scaled),
        ("Stacking ensemble", stacking_model, X_train_scaled, X_test_scaled),
    ]
)

# 各モデルの予測値、評価結果、学習済みモデルを保存する入れ物を作ります。
model_predictions = {}
model_score_rows = []

trained_models = {}

# すべての候補モデルを学習し、テストデータで予測性能を評価します。
for model_name, candidate_model, train_features, test_features in model_specs:
    candidate_model.fit(train_features, y_train)
    candidate_pred = candidate_model.predict(test_features)
    model_predictions[model_name] = candidate_pred
    trained_models[model_name] = candidate_model

    model_score_rows.append(
        {
            "model": model_name,
            "MAE": mean_absolute_error(y_test, candidate_pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, candidate_pred)),
            "R2": r2_score(y_test, candidate_pred),
        }
    )

# RMSEが小さい順にモデル比較表を作ります。
model_scores = (
    pd.DataFrame(model_score_rows)
    .sort_values("RMSE", ascending=True)
    .reset_index(drop=True)
)

# モデル比較結果を表として表示します。
print("Hold-Out 検証のモデル比較")
display(model_scores.round({"MAE": 3, "RMSE": 3, "R2": 3}))

# モデルごとのRMSEを棒グラフで可視化します。
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=model_scores, x="RMSE", y="model", color="#4C78A8", ax=ax)
ax.set_title("Regression model comparison by RMSE")
ax.set_xlabel("RMSE (lower is better)")
ax.set_ylabel("Model")
plt.show()

# RMSEが最も小さいモデルを、以降のAD解析に使うモデルとして選びます。
selected_model_name = model_scores.loc[0, "model"]
model = trained_models[selected_model_name]
y_pred = model_predictions[selected_model_name]

# 選択したモデルの予測性能を計算します。
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

# 選択モデル名と性能指標を表示します。
print(f"以降のAD解析に使うモデル: {selected_model_name}")
print("Hold-Out 検証の予測性能（選択モデル）")
print(f"MAE : {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R^2 : {r2:.3f}")


## スライド対応: スライド1

- 距離計算では特徴量のスケール差が効くため、AD判定には標準化済み説明変数を使う。
- 回帰モデルは単体モデルだけでなくVoting/Stackingも試し、Hold-OutのRMSEが最小のモデルをAD解析の予測値として使う。

## 4. 2種類のADしきい値を計算

1つ目はスライドに沿ったkNN平均距離ADです。2つ目はMahalanobis距離ADです。Mahalanobis距離は特徴量間の相関も考慮するため、単純なEuclidean距離とは違う観点で「学習データらしさ」を評価できます。

- kNN-AD: 標準化説明変数空間のEuclidean距離、k=5、学習データの5近傍平均距離の95パーセンタイルをしきい値
- Mahalanobis-AD: 標準化説明変数空間のMahalanobis距離、学習データ距離の95パーセンタイルをしきい値

kNN-ADでは、学習データ同士で近傍を取ると自分自身の距離0が含まれるため、`k + 1` 近傍を取得して自分自身を除外します。

In [ ]:
# kNN-ADで使う近傍数を指定します。
k = 5

# 学習データ側の距離分布の何パーセンタイルをしきい値にするか指定します。
percentile = 95

# 距離の計算方法としてユークリッド距離を使います。
metric = "euclidean"

# 学習データの近傍を探すNearestNeighborsモデルを作ります。
# 学習データ同士では自分自身も近傍に入るため、k + 1個を取得します。
nn = NearestNeighbors(n_neighbors=k + 1, metric=metric)

# 標準化した学習データを使って近傍探索モデルを学習します。
nn.fit(X_train_scaled)

# 訓練データ同士: 0列目は自分自身への距離0なので除外します。
train_distances, _ = nn.kneighbors(X_train_scaled)
train_knn_mean_distance = train_distances[:, 1:].mean(axis=1)

# 学習データのk近傍平均距離の95パーセンタイルをkNN-ADしきい値にします。
knn_ad_threshold = np.percentile(train_knn_mean_distance, percentile)

# テストデータ: 自分自身は含まれないので先頭k列をそのまま使用します。
test_distances, _ = nn.kneighbors(X_test_scaled)
test_knn_mean_distance = test_distances[:, :k].mean(axis=1)

# テストデータのk近傍平均距離がしきい値以下ならkNN-AD内と判定します。
is_in_knn_ad = test_knn_mean_distance <= knn_ad_threshold

# Mahalanobis距離AD: Ledoit-Wolfで共分散推定を安定化します。
covariance_model = LedoitWolf().fit(X_train_scaled)

# 学習データとテストデータのMahalanobis距離を計算します。
train_mahalanobis_distance = covariance_model.mahalanobis(X_train_scaled)
test_mahalanobis_distance = covariance_model.mahalanobis(X_test_scaled)

# 学習データのMahalanobis距離の95パーセンタイルをしきい値にします。
mahalanobis_ad_threshold = np.percentile(train_mahalanobis_distance, percentile)

# テストデータのMahalanobis距離がしきい値以下ならMahalanobis-AD内と判定します。
is_in_mahalanobis_ad = test_mahalanobis_distance <= mahalanobis_ad_threshold

# 2種類のAD判定が一致しているか確認します。
ad_agreement = is_in_knn_ad == is_in_mahalanobis_ad

# 2種類のAD判定を組み合わせて、予測信頼度をhigh、medium、lowに分けます。
reliability_label = np.select(
    [is_in_knn_ad & is_in_mahalanobis_ad, is_in_knn_ad | is_in_mahalanobis_ad],
    ["high", "medium"],
    default="low",
)

# AD判定に使った設定と、テストデータのAD内外の数を表示します。
print(f"k = {k}")
print(f"距離指標 = {metric}")
print(f"kNN-ADしきい値 ({percentile}パーセンタイル) = {knn_ad_threshold:.3f}")
print(f"kNN-AD内のテストデータ数: {int(is_in_knn_ad.sum())}")
print(f"kNN-AD外のテストデータ数: {int((~is_in_knn_ad).sum())}")
print(f"Mahalanobis-ADしきい値 ({percentile}パーセンタイル) = {mahalanobis_ad_threshold:.3f}")
print(f"Mahalanobis-AD内のテストデータ数: {int(is_in_mahalanobis_ad.sum())}")
print(f"Mahalanobis-AD外のテストデータ数: {int((~is_in_mahalanobis_ad).sum())}")
print(f"2種類のAD判定が一致したテストデータ数: {int(ad_agreement.sum())} / {len(ad_agreement)}")


## スライド対応: スライド1

- スライド1の手順に沿ったkNN-ADを主判定として使う。
- 独自性として、特徴量間の相関も見るMahalanobis-ADを追加し、2つのAD判定が一致するかを確認する。
- kNN-ADとMahalanobis-ADの両方でAD内なら信頼度 `high`、片方だけAD内なら `medium`、両方AD外なら `low` とする。

## 5. テストデータごとのAD判定と誤差

In [ ]:
# テストデータごとに、実測値、予測値、誤差、AD判定を1つの表にまとめます。
results = pd.DataFrame(
    {
        "actual_MEDV": y_test,
        "predicted_MEDV": y_pred,
        "residual": y_test.to_numpy() - y_pred,
        "abs_error": np.abs(y_test.to_numpy() - y_pred),
        "knn_mean_distance": test_knn_mean_distance,
        "mahalanobis_distance": test_mahalanobis_distance,
        "kNN_AD": np.where(is_in_knn_ad, "inside", "outside"),
        "Mahalanobis_AD": np.where(is_in_mahalanobis_ad, "inside", "outside"),
        "reliability": reliability_label,
    },
    index=y_test.index,
)

# 信頼度の表示順を high、medium、low の順に固定します。
results["reliability"] = pd.Categorical(results["reliability"], categories=["high", "medium", "low"], ordered=True)

# 信頼度と絶対誤差で並べ替えて、注意して確認したいサンプルを表示します。
display(results.sort_values(["reliability", "abs_error"], ascending=[False, False]).head(10))


In [ ]:
# 信頼度ごとに、サンプル数、誤差、平均距離を集計します。
summary_by_reliability = (
    results.groupby("reliability")
    .agg(
        n=("actual_MEDV", "size"),
        MAE=("abs_error", "mean"),
        RMSE=("residual", lambda x: np.sqrt(np.mean(x**2))),
        mean_knn_distance=("knn_mean_distance", "mean"),
        mean_mahalanobis_distance=("mahalanobis_distance", "mean"),
    )
    .sort_index()
)

# kNN-ADとMahalanobis-ADの判定がどのように対応しているかをクロス集計します。
ad_cross_table = pd.crosstab(results["kNN_AD"], results["Mahalanobis_AD"])

# 信頼度別の集計表と、2種類のAD判定のクロス集計表を表示します。
display(summary_by_reliability)
display(ad_cross_table)


## スライド対応: スライド1

- テストデータ各点に、実測値、予測値、残差、絶対誤差、kNN平均距離、AD内外ラベルを付ける。
- AD内外ごとにMAE/RMSEを集計し、AD外の予測信頼性を確認する。

## 6. 可視化1: 2種類のAD距離分布としきい値

In [ ]:
# kNN距離とMahalanobis距離の分布を横並びで表示する図を作ります。
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 学習データのk近傍平均距離の分布をヒストグラムで表示します。
sns.histplot(train_knn_mean_distance, bins=30, kde=True, color="#4C78A8", ax=axes[0])

# kNN-ADのしきい値を赤い破線で表示します。
axes[0].axvline(knn_ad_threshold, color="#D62728", linestyle="--", linewidth=2, label=f"threshold = {knn_ad_threshold:.3f}")
axes[0].set_title("kNN mean distance AD")
axes[0].set_xlabel("Mean distance to 5 nearest neighbors")
axes[0].set_ylabel("Count")
axes[0].legend()

# 学習データのMahalanobis距離の分布をヒストグラムで表示します。
sns.histplot(train_mahalanobis_distance, bins=30, kde=True, color="#6A4C93", ax=axes[1])

# Mahalanobis-ADのしきい値を赤い破線で表示します。
axes[1].axvline(mahalanobis_ad_threshold, color="#D62728", linestyle="--", linewidth=2, label=f"threshold = {mahalanobis_ad_threshold:.3f}")
axes[1].set_title("Mahalanobis distance AD")
axes[1].set_xlabel("Mahalanobis distance")
axes[1].set_ylabel("Count")
axes[1].legend()

# 図同士が重ならないように余白を調整します。
fig.tight_layout()

# 図を表示します。
plt.show()


## スライド対応: スライド2 可視化例1

- スライド2の例1を拡張し、kNN平均距離とMahalanobis距離を横並びで可視化する。
- 赤破線より右側は、それぞれの距離基準で学習データから離れた領域。
- kNNは局所的な疎密、Mahalanobisは全体の共分散構造から見た外れ具合を表す。

## 7. 可視化2: テストデータのk近傍距離と予測誤差

横軸がkNN-AD判定に使ったテストデータの5近傍平均距離、縦軸が絶対誤差です。点の色は、kNN-ADとMahalanobis-ADを組み合わせた信頼度です。赤い破線より右側はkNN-AD外です。

In [ ]:
# テストデータのk近傍平均距離と絶対誤差の関係を見る図を作ります。
fig, ax = plt.subplots()

# 横軸をk近傍平均距離、縦軸を絶対誤差として散布図を描きます。
# 色は信頼度、マーカーの形はkNN-ADの内外を表します。
sns.scatterplot(
    data=results,
    x="knn_mean_distance",
    y="abs_error",
    hue="reliability",
    style="kNN_AD",
    palette={"high": "#2E7D32", "medium": "#F2A93B", "low": "#C62828"},
    s=70,
    ax=ax,
)

# kNN-ADのしきい値を赤い破線で表示します。
ax.axvline(knn_ad_threshold, color="#D62728", linestyle="--", linewidth=2, label="kNN-AD threshold")
ax.set_title("Prediction error vs. kNN mean distance")
ax.set_xlabel("Mean distance to 5 nearest train samples")
ax.set_ylabel("Absolute prediction error")

# 凡例の重複を取り除いて表示します。
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title=None)

# 図を表示します。
plt.show()


## スライド対応: スライド2 可視化例2

- スライド2の例2に対応する図。
- 横軸がテストデータの5近傍平均距離、縦軸が絶対予測誤差。
- 赤破線より右側がAD外。AD外は学習データから離れているため、予測信頼性を低めに扱う。

## 8. 予測値の信頼性の可視化

実測値と予測値の散布図を信頼度で色分けします。2種類のAD判定の両方で外側に出る点は、学習データから離れた外挿領域にあるため、予測値の信頼性が相対的に低いと考えます。

In [ ]:
# 実測値と予測値の対応を見る散布図を作ります。
fig, ax = plt.subplots()

# 横軸を実測値、縦軸を予測値として散布図を描きます。
# 色は信頼度、マーカーの形はMahalanobis-ADの内外を表します。
sns.scatterplot(
    data=results,
    x="actual_MEDV",
    y="predicted_MEDV",
    hue="reliability",
    style="Mahalanobis_AD",
    palette={"high": "#2E7D32", "medium": "#F2A93B", "low": "#C62828"},
    s=70,
    ax=ax,
)

# 実測値と予測値が完全に一致する理想線を描くため、軸の最小値と最大値を求めます。
min_value = min(results["actual_MEDV"].min(), results["predicted_MEDV"].min())
max_value = max(results["actual_MEDV"].max(), results["predicted_MEDV"].max())

# 理想線を点線で描きます。点がこの線に近いほど予測が良いことを表します。
ax.plot([min_value, max_value], [min_value, max_value], color="#333333", linestyle=":", label="ideal")
ax.set_title("Actual vs. predicted MEDV colored by AD")
ax.set_xlabel("Actual MEDV")
ax.set_ylabel("Predicted MEDV")

# 実測値と予測値を同じ縮尺で比較できるように、縦横比を1:1にします。
ax.set_aspect("equal", adjustable="box")

# 凡例の重複を取り除いて表示します。
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title=None)

# 図を表示します。
plt.show()


## スライド対応: スライド3

- スライド3のように、AD内外で点を色分けして予測信頼性を考えるための図。
- 対角線に近いほど予測が良い。AD外点は学習データから離れているため、信頼性を低めに見る。
- AD計算は可視化用の低次元空間ではなく、元の13次元標準化説明変数空間で行っている。

## 9. 低信頼サンプルの確認

In [ ]:
# 両方のAD判定で外側になった低信頼サンプルを、絶対誤差が大きい順に取り出します。
low_reliability = results[results["reliability"] == "low"].sort_values("abs_error", ascending=False)

# 片方のAD判定だけで外側になった中信頼サンプルも、絶対誤差が大きい順に取り出します。
medium_reliability = results[results["reliability"] == "medium"].sort_values("abs_error", ascending=False)

# 低信頼サンプルがない場合は、中信頼サンプルを表示して確認します。
if low_reliability.empty:
    print("このHold-Out分割では、kNN-ADとMahalanobis-ADの両方でAD外となる低信頼サンプルはありませんでした。")
    print("片方のADだけで外側になったmedium信頼サンプルを確認します。")
    display(medium_reliability)

# 低信頼サンプルがある場合は、その一覧と平均絶対誤差を表示します。
else:
    display(low_reliability)
    print(
        f"低信頼サンプルのMAEは {low_reliability['abs_error'].mean():.3f} で、"
        f"全テストデータのMAE {mae:.3f} と比較して予測信頼性を確認できます。"
    )


## スライド対応: スライド1-3

- kNN-ADとMahalanobis-ADの両方でAD外になったサンプルを低信頼サンプルとして確認する。
- 片方だけAD外になったサンプルはmedium信頼として扱い、低信頼とは分けて解釈する。
- 低信頼は「必ず誤差が大きい」ではなく、「2つの距離基準で学習データから離れているため信頼性を低めに扱う」判定。

## 10. まとめ

- `MEDV` を目的変数としてHold-Out検証による回帰モデルを作成した。
- `DummyRegressor`、線形回帰、正則化線形モデル、kNN、SVR、決定木、ランダムフォレスト、Extra Trees、勾配ブースティング系モデルなどを同じデータ分割で比較した。
- 比較指標には `MAE`、`RMSE`、`R^2` を使用し、以降のAD解析では `RMSE` が最小だったモデルの予測値を用いた。
- 標準化した説明変数空間でkNN平均距離ADとMahalanobis距離ADを計算し、それぞれ学習データの95パーセンタイルをしきい値とした。
- 2種類のAD判定を組み合わせ、両方AD内を `high`、片方のみAD内を `medium`、両方AD外を `low` とした。
- `low` の点は2つの距離基準で学習データから離れた外挿領域にあるため、予測値の信頼性が相対的に低いサンプルとして扱える。


## スライド対応: まとめ

- 指定スライドのAD手順に沿って、kNN平均距離と95パーセンタイルしきい値でAD内外を判定した。
- 可視化は、スライド2の2例に加えて、実測値 vs 予測値をAD内外で色分けした図も用意した。
- 結果として、AD外サンプルは学習データから離れた予測信頼性の低い点として解釈できる。